# Tools and RAG



## Defining problem
LLM can provide answer based on the following information:
- Trained data
- Licensed data
- Publically available data

**What if I want to ask question based on information which is available in my knowledge base?**
- examples:
    - What should be the maintenance strategy for pump-01
    - Create maintenance notification for equipment pump-01

- Solution:
    - Connect our Knowledge base to LLM.
    - Knowledge base can be either:
        - Database
        - Documents
        - Internal portal links which is not publicly accessible.

### Two ways of connecting our knowledge base to LLM
    - Tools
    - Resources( RAG )



### Difference between RAG and Tools
| Aspect | RAG (Retrieval-Augmented Generation) | Tools/Function Calling |
  |--------|--------------------------------------|------------------------|
  | **Purpose** | Provide knowledge/context to LLM | Allow LLM to perform actions or fetch real-time data |
  | **Timing** | Preprocessing before LLM call | During/after LLM reasoning |
  | **LLM Involvement** | Passive recipient of context | Active decision maker to use tools |
  | **Data Flow** | Knowledge base → Prompt → LLM | LLM → Tool call → Execute → LLM |
  | **Data Type** | Static/historical knowledge | Dynamic/real-time data |
  | **State** | Read-only retrieval | Can modify state/perform actions |
  | **Use Case Example** | "What does the manual say about pump maintenance?" | "What is the current temperature of the pump?" |
  | **Knowledge Source** | Pre-indexed documents, embeddings, vector database | Live databases, APIs, sensors, external systems |
  | **When Data Changes** | Requires re-indexing/re-embedding | Automatically reflects latest data |
  | **Latency** | Vector search + LLM inference | Function execution + LLM inference (may be slower) |
  | **Cost** | Higher storage (embeddings), one LLM call | Lower storage, may need multiple LLM calls |
  | **LLM Control** | Cannot choose which documents to retrieve | Can choose which tools to call |



## Tools in action

In [61]:
# Import required libraries
import os
from dotenv import load_dotenv
from openai import OpenAI
import glob
import fitz  # PyMuPDF
import gradio as gr
import sqlite3
import random
import json
from datetime import datetime, timedelta
from langchain_community.chat_models import ChatOllama
from langchain_community.embeddings import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [62]:
## Create a SQL Lite data base

DB = "maintenance.db"

## Create equipment and maintenance plan tables
# Connect database
conn = sqlite3.connect("maintenance.db")
cursor = conn.cursor()

# Enable foreign keys
cursor.execute("PRAGMA foreign_keys = ON")

# -------------------------
# Create Equipment Table
# -------------------------
cursor.execute("""
CREATE TABLE IF NOT EXISTS equipment (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    serial_number TEXT UNIQUE,
    location TEXT,
    purchase_date TEXT,
    status TEXT
)
""")

# -------------------------
# Create Maintenance Plan Table
# -------------------------
cursor.execute("""
CREATE TABLE IF NOT EXISTS maintenance_plan (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    equipment_id INTEGER,
    maintenance_type TEXT,
    frequency_days INTEGER,
    last_maintenance TEXT,
    next_due TEXT,
    FOREIGN KEY (equipment_id)
        REFERENCES equipment(id)
        ON DELETE CASCADE
)
""")

# -------------------------
# Sample Data Pools
# -------------------------
equipment_types = [
    "Pump", "Compressor", "Generator", "Motor",
    "Conveyor", "Boiler", "Fan", "Chiller"
]

locations = ["Plant A", "Plant B", "Warehouse", "Unit 1", "Unit 2"]
statuses = ["Active", "Inactive"]
maintenance_types = ["Preventive", "Inspection", "Calibration"]

# -------------------------
# Helper Function
# -------------------------
def random_date(start_year=2020, end_year=2024):
    start = datetime(start_year, 1, 1)
    end = datetime(end_year, 12, 31)
    delta = end - start
    random_days = random.randint(0, delta.days)
    return start + timedelta(days=random_days)

# -------------------------
# Insert 100 Equipments
# -------------------------
equipment_ids = []

for i in range(1, 101):
    name = f"{random.choice(equipment_types)}-{i}"
    serial = f"SN-{1000+i}"
    location = random.choice(locations)
    purchase_date = random_date().strftime("%Y-%m-%d")
    status = random.choice(statuses)

    cursor.execute("""
        INSERT INTO equipment
        (name, serial_number, location, purchase_date, status)
        VALUES (?, ?, ?, ?, ?)
    """, (name, serial, location, purchase_date, status))

    equipment_ids.append(cursor.lastrowid)

# -------------------------
# Insert Maintenance Plans
# -------------------------
for eq_id in equipment_ids:
    frequency = random.choice([7, 15, 30, 60, 90])

    last_maintenance = datetime.now() - timedelta(days=random.randint(1, 60))
    next_due = last_maintenance + timedelta(days=frequency)

    cursor.execute("""
        INSERT INTO maintenance_plan
        (equipment_id, maintenance_type, frequency_days,
         last_maintenance, next_due)
        VALUES (?, ?, ?, ?, ?)
    """, (
        eq_id,
        random.choice(maintenance_types),
        frequency,
        last_maintenance.strftime("%Y-%m-%d"),
        next_due.strftime("%Y-%m-%d")
    ))

# -------------------------
# Commit & Verify
# -------------------------
# Save changes
conn.commit()

conn.close()

In [63]:
# Connect database
conn = sqlite3.connect("maintenance.db")
cursor = conn.cursor()

cursor.execute("""
SELECT e.id, e.name, e.location, m.maintenance_type, m.next_due
FROM equipment e
JOIN maintenance_plan m
ON e.id = m.equipment_id
""")

rows = cursor.fetchall()

for row in rows:
    print(row)

conn.close()

(1, 'Pump-1', 'Unit 2', 'Calibration', '2026-06-15')
(2, 'Motor-2', 'Unit 1', 'Inspection', '2026-04-07')
(3, 'Compressor-3', 'Plant B', 'Inspection', '2026-04-07')
(4, 'Boiler-4', 'Unit 1', 'Calibration', '2026-03-24')
(5, 'Chiller-5', 'Unit 1', 'Preventive', '2026-03-26')
(6, 'Conveyor-6', 'Plant A', 'Preventive', '2026-02-28')
(7, 'Fan-7', 'Warehouse', 'Inspection', '2026-04-09')
(8, 'Chiller-8', 'Plant A', 'Inspection', '2026-04-06')
(9, 'Compressor-9', 'Plant A', 'Inspection', '2026-03-11')
(10, 'Generator-10', 'Unit 2', 'Preventive', '2026-04-01')
(11, 'Motor-11', 'Warehouse', 'Inspection', '2026-03-18')
(12, 'Generator-12', 'Plant B', 'Preventive', '2026-05-01')
(13, 'Generator-13', 'Unit 2', 'Calibration', '2026-05-05')
(14, 'Generator-14', 'Unit 2', 'Inspection', '2026-05-14')
(15, 'Chiller-15', 'Plant B', 'Preventive', '2026-04-30')
(16, 'Boiler-16', 'Unit 2', 'Inspection', '2026-03-17')
(17, 'Generator-17', 'Plant B', 'Inspection', '2026-05-17')
(18, 'Fan-18', 'Unit 1', 'Pre

In [64]:
# Creating tools method
def get_maintenance_type(equipment):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
    cursor.execute(' SELECT m.maintenance_type FROM equipment e JOIN maintenance_plan m ON e.id = m.equipment_id where e.name = ? ', (equipment,))
    result = cursor.fetchone()
    return f"Maintenance type for {equipment} is ${result[0]}" if result else "No Maintenance type available for this equipment"

In [65]:
# Creating tools function
maintenance_function = {
    "name": "get_maintenance_type",
    "description": "Get the maintenance type for given equipment.",
    "parameters": {
        "type": "object",
        "properties": {
            "equipment": {
                "type": "string",
                "description": "This is the equipment for which he need to find maintenance type",
            },
        },
        "required": ["equipment"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": maintenance_function}]
tools

[{'type': 'function',
  'function': {'name': 'get_maintenance_type',
   'description': 'Get the maintenance type for given equipment.',
   'parameters': {'type': 'object',
    'properties': {'equipment': {'type': 'string',
      'description': 'This is the equipment for which he need to find maintenance type'}},
    'required': ['equipment'],
    'additionalProperties': False}}}]

In [66]:

def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        print(json.loads(tool_call.function.arguments))
        if tool_call.function.name == "get_maintenance_type":
            arguments = json.loads(tool_call.function.arguments)
  
            equipment = arguments.get('equipment')
            maintenance_type = get_maintenance_type(equipment)
            responses.append({
                "role": "tool",
                "content": maintenance_type,
                "tool_call_id": tool_call.id
            })
    return responses

In [67]:

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

openai = OpenAI()

API key found and looks good so far!


In [68]:
system_message = """
You are a helpful assistant for a manufacturing plant.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages, tools=tools)
    
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.


{'equipment': 'Chiller-8'}


In [69]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='fhsdjkfhsdj')

In [70]:
system_message = """
You are a helpful assistant for a manufacturing plant.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model="gemma3:4b", messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = ollama.chat.completions.create(model="gemma3:4b", messages=messages, tools=tools)
    
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/Users/I072374/Sharath/Projects/AILearning/AILearning/LLM/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/I072374/Sharath/Projects/AILearning/AILearning/LLM/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/I072374/Sharath/Projects/AILearning/AILearning/LLM/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2191, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/I072374/Sharath/Projects/AILearning/AILearning/LLM/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1696, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^


In [71]:
system_message = """
You are a helpful assistant for a manufacturing plant.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model="llama3.1", messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = ollama.chat.completions.create(model="llama3.1", messages=messages, tools=tools)
    
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.


## Let's do some RAGging

RAG combines retrieval and generation:
1. **User asks a question**
2. **System searches knowledge base** (documents, databases)
3. **Retrieved content is added to prompt**
4. **LLM generates answer** using the retrieved context


## Key Concepts


### Types of LLMs
- Auto regressive LLMs
    - GPT
    - Claude
    - Works on token to token
- Auto Embedding LLMs/ Auto Encoding LLMs
    - EmbeddingGemma
    - Nomic-Embed-Text
    - Works on meaning


### Vector Embeddings
- **Embeddings** convert text into numerical vectors
- Similar meanings = similar vectors
- Example: "Hello" → [0.12, -0.34, 0.56, ...] (768 dimensions)
- We need special kind of LLM called AutoEmbedding LLM to create vector embedding.


### Vector Database
- Stores and searches embeddings efficiently
- Finds semantically similar content
- We use **ChromaDB** in this example

In [78]:
# Configuration
OLLAMA_BASE_URL = "http://localhost:11434"
EMBEDDING_MODEL = "nomic-embed-text"
LLM_MODEL = "llama3.2"  # 3B model - good balance of speed and quality
VECTOR_DB_PATH = "./vector_db_improved2"
KNOWLEDGE_PATH = "knowledge/*.pdf"

# RAG settings
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
TOP_K_RESULTS = 4  # Number of relevant chunks to retrieve

## Test Embedding Model

Let's verify our embedding model works correctly.

In [74]:
# Initialize embedding model
embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL, base_url=OLLAMA_BASE_URL)

# Test embedding
test_text = "Hello, this is a test"
test_vector = embeddings.embed_query(test_text)

print(f"Text: {test_text}")
print(f"\nEmbedding vector (first 10 values):")
print(test_vector[:10])
print(f"\nVector dimensions: {len(test_vector)}")

Text: Hello, this is a test

Embedding vector (first 10 values):
[0.7247158885002136, 0.3518443703651428, -3.8605563640594482, -0.365138977766037, 1.8804348707199097, -0.3544164001941681, 0.4602046012878418, -0.4450487196445465, 0.9083202481269836, -1.1460468769073486]

Vector dimensions: 768


## Step 1: Load Documents

We'll load PDF documents from the knowledge folder.

In [75]:
def load_pdf_documents(pattern: str) -> list[Document]:
    """
    Load PDF files and convert them to LangChain Documents.
    
    Args:
        pattern: Glob pattern for PDF files (e.g., "knowledge/*.pdf")
        
    Returns:
        List of Document objects with page content and metadata
    """
    pdf_files = glob.glob(pattern)
    documents = []
    
    print(f"Found {len(pdf_files)} PDF files")
    
    for pdf_path in pdf_files:
        print(f"Reading: {pdf_path}")
        
        try:
            doc = fitz.open(pdf_path)
            
            for page_num, page in enumerate(doc):
                text = page.get_text()
                
                # Skip empty pages
                if text.strip():
                    documents.append(
                        Document(
                            page_content=text,
                            metadata={
                                "source": os.path.basename(pdf_path),
                                "page": page_num + 1,
                                "file_path": pdf_path
                            }
                        )
                    )
            
            doc.close()
            
        except Exception as e:
            print(f"Error reading {pdf_path}: {e}")
    
    print(f"\nLoaded {len(documents)} pages from {len(pdf_files)} files")
    return documents

# Load documents
documents = load_pdf_documents(KNOWLEDGE_PATH)

Found 3 PDF files
Reading: knowledge/PWI_Pump.pdf
Reading: knowledge/compressed_air_manual.pdf
Reading: knowledge/heat_pump.pdf

Loaded 201 pages from 3 files


## Step 2: Split Documents into Chunks

Large documents need to be split into smaller chunks:
- **Chunk size**: 1000 characters (optimal for most use cases)
- **Overlap**: 200 characters (maintains context between chunks)

In [76]:
# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]  # Try to split on paragraphs first
)

chunks = text_splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks")

# Show example chunk
if chunks:
    print("\nExample chunk:")
    print(f"Source: {chunks[0].metadata['source']}, Page: {chunks[0].metadata['page']}")
    print(f"Content preview: {chunks[0].page_content[:200]}...")

Split into 538 chunks

Example chunk:
Source: PWI_Pump.pdf, Page: 1
Content preview: 8885. Monroe St.  Houston, Texas USA 77061 
 
Phone: (713) 956-2002 
Fax(713) 956-2141 
 
 
STDS-PWI-IOM-TM-0.0 
 
1/26 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
INSTALLATION OPERATION AND 
MAINTENANCE MANUAL ...


## Step 3: Create Vector Store

We'll create embeddings for all chunks and store them in ChromaDB.

In [79]:
# Remove existing vector store if it exists
if os.path.exists(VECTOR_DB_PATH):
    import shutil
    shutil.rmtree(VECTOR_DB_PATH)
    print("Removed existing vector store")

# Create new vector store
print("Creating vector store... (this may take a minute)")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=VECTOR_DB_PATH
)

# Verify vector store
collection = vectorstore._collection
count = collection.count()
sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)

print(f"\n✓ Vector store created successfully!")
print(f"  - Total vectors: {count:,}")
print(f"  - Dimensions: {dimensions:,}")
print(f"  - Storage path: {VECTOR_DB_PATH}")

Creating vector store... (this may take a minute)

✓ Vector store created successfully!
  - Total vectors: 538
  - Dimensions: 768
  - Storage path: ./vector_db_improved2


## Step 4: Test Retrieval

Let's test if we can retrieve relevant documents.

In [81]:
# Create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K_RESULTS}
)

# Test queries
test_queries = [
    #"What are the maintenance procedures?"
    "How to perform pump inspection?"
    #"What is the lubrication schedule?"
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    
    docs = retriever.invoke(query)
    print(f"Retrieved {len(docs)} relevant chunks:\n")
    
    for i, doc in enumerate(docs, 1):
        print(f"{i}. Source: {doc.metadata['source']}, Page: {doc.metadata['page']}")
        print(f"   Preview: {doc.page_content[:150].strip()}...")
        print()


Query: How to perform pump inspection?
Retrieved 4 relevant chunks:

1. Source: PWI_Pump.pdf, Page: 10
   Preview: during pump operation. 
 
Freezing 
 
During cold weather when the pump is not in operation, the pump should be drained to prevent 
the liquid inside...

2. Source: PWI_Pump.pdf, Page: 11
   Preview: 8885. Monroe St.  Houston, Texas USA 77061 
 
Phone: (713) 956-2002 
Fax(713) 956-2141 
 
 
STDS-PWI-IOM-TM-0.0 
 
11/26 
Section 6 - Starting the Pum...

3. Source: PWI_Pump.pdf, Page: 10
   Preview: 8885. Monroe St.  Houston, Texas USA 77061 
 
Phone: (713) 956-2002 
Fax(713) 956-2141 
 
 
STDS-PWI-IOM-TM-0.0 
 
10/26 
Section 5 – Operation 
Opera...

4. Source: PWI_Pump.pdf, Page: 19
   Preview: threads, cylindrical fits and gasket surfaces. Any burred edge must be removed before part is 
installed into the pump. Coat all parts with light oil...



## Step 5: Initialize LLM

We'll use Llama 3.2 (3B) for generation - it's fast and produces quality results.

In [82]:
# Initialize LLM
llm = ChatOllama(
    model=LLM_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=0.1,  # Lower temperature for more focused answers
)

# Test LLM
print("Testing LLM...")
response = llm.invoke([HumanMessage(content="Say hello in one sentence.")])
print(f"LLM response: {response.content}")

Testing LLM...
LLM response: Hello!


## Step 6: Create RAG Chain

Now we'll create a complete RAG pipeline.

In [83]:
# Create prompt template
prompt_template = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant for a manufacturing plant maintenance system.
    
Use the following context to answer the user's question. If you cannot find the answer in the context, say so clearly.

Guidelines:
- Be accurate and specific
- Cite the source document and page number when possible
- If you're not sure, say "I don't have enough information to answer that"
- Keep answers concise but complete

Context:
{context}
"""),
    ("human", "{question}")
])

def format_docs(docs: list[Document]) -> str:
    """
    Format retrieved documents into a context string.
    """
    formatted = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata['source']
        page = doc.metadata['page']
        content = doc.page_content.strip()
        formatted.append(f"[Source {i}: {source}, Page {page}]\n{content}")
    return "\n\n" + "\n\n".join(formatted)

def answer_question(question: str) -> dict:
    """
    Answer a question using RAG.
    
    Returns:
        Dictionary with answer, sources, and retrieved documents
    """
    try:
        # Retrieve relevant documents
        docs = retriever.invoke(question)
        
        if not docs:
            return {
                "answer": "I couldn't find any relevant information in the knowledge base.",
                "sources": [],
                "retrieved_docs": []
            }
        
        # Format context
        context = format_docs(docs)
        
        # Create prompt
        messages = prompt_template.format_messages(
            context=context,
            question=question
        )
        
        # Get LLM response
        response = llm.invoke(messages)
        
        # Extract sources
        sources = [{
            "source": doc.metadata['source'],
            "page": doc.metadata['page']
        } for doc in docs]
        
        return {
            "answer": response.content,
            "sources": sources,
            "retrieved_docs": docs
        }
    
    except Exception as e:
        return {
            "answer": f"An error occurred: {str(e)}",
            "sources": [],
            "retrieved_docs": []
        }

print("RAG chain created successfully!")

RAG chain created successfully!


## Step 7: Test RAG System

Let's test the complete RAG system with various questions.

In [ ]:
# Test questions
test_questions = [
    "What are the steps for pump maintenance?",
    "How should I lubricate the equipment?",
    "What should I check during inspection?",
    "Tell me about mechanical seal maintenance"
]

for question in test_questions:
    print(f"\n{'='*80}")
    print(f"Question: {question}")
    print(f"{'='*80}\n")
    
    result = answer_question(question)
    
    print("Answer:")
    print(result['answer'])
    
    print("\nSources:")
    for source in result['sources']:
        print(f"  - {source['source']}, Page {source['page']}")
    
    print()

## Step 8: Create Interactive Chat Interface

We'll create a Gradio interface for easy interaction.

In [84]:
def chat_interface(message: str, history: list) -> str:
    """
    Chat interface function for Gradio.
    """
    result = answer_question(message)
    
    # Format response with sources
    response = result['answer']
    
    if result['sources']:
        response += "\n\n**Sources:**\n"
        for source in result['sources']:
            response += f"- {source['source']}, Page {source['page']}\n"
    
    return response

# Create Gradio interface
demo = gr.ChatInterface(
    fn=chat_interface,
    title="🏭 Manufacturing Plant RAG Assistant",
    description="Ask questions about equipment maintenance and procedures. The system will search through technical manuals to provide accurate answers.",
    examples=[
        "What are the maintenance procedures for pumps?",
        "How often should I perform lubrication?",
        "What should I check during equipment inspection?",
        "Tell me about mechanical seals"
    ],
    theme=gr.themes.Soft(),
    type="messages"
)

# Launch interface
demo.launch(share=False)

* Running on local URL:  http://127.0.0.1:7875
* To create a public link, set `share=True` in `launch()`.


## Advanced: Evaluate RAG Quality

Let's create some metrics to evaluate our RAG system.

In [ ]:
def evaluate_retrieval(question: str, expected_keywords: list[str]) -> dict:
    """
    Evaluate retrieval quality by checking if expected keywords appear in retrieved docs.
    """
    docs = retriever.invoke(question)
    
    combined_text = " ".join([doc.page_content.lower() for doc in docs])
    
    found_keywords = [kw for kw in expected_keywords if kw.lower() in combined_text]
    
    return {
        "question": question,
        "docs_retrieved": len(docs),
        "expected_keywords": expected_keywords,
        "found_keywords": found_keywords,
        "keyword_coverage": len(found_keywords) / len(expected_keywords) if expected_keywords else 0
    }

# Test cases
test_cases = [
    ("How to maintain pumps?", ["pump", "maintenance", "lubrication"]),
    ("What is the inspection procedure?", ["inspection", "check", "procedure"]),
    ("Tell me about mechanical seals", ["seal", "mechanical", "leakage"])
]

print("Retrieval Quality Evaluation:\n")
for question, keywords in test_cases:
    result = evaluate_retrieval(question, keywords)
    print(f"Question: {result['question']}")
    print(f"  Documents retrieved: {result['docs_retrieved']}")
    print(f"  Keyword coverage: {result['keyword_coverage']:.1%}")
    print(f"  Found: {result['found_keywords']}")
    print()

## Summary and Next Steps

### What We Built:
1. ✅ Document loader for PDF files
2. ✅ Text chunking with optimal overlap
3. ✅ Vector embeddings using Ollama
4. ✅ ChromaDB vector store
5. ✅ Semantic search/retrieval
6. ✅ LLM-based answer generation
7. ✅ Interactive chat interface
8. ✅ Source citation

### Improvements You Can Make:
1. **Better chunking**: Use semantic chunking instead of fixed-size chunks
2. **Reranking**: Add a reranker to improve retrieval quality
3. **Hybrid search**: Combine semantic search with keyword search
4. **Query expansion**: Rephrase queries for better retrieval
5. **Caching**: Cache common queries for faster responses
6. **Evaluation**: Add automated evaluation metrics
7. **Multi-query**: Generate multiple queries for comprehensive retrieval

### Model Recommendations:
- **Best quality**: `llama3.1:8b` or `mixtral:8x7b`
- **Best speed**: `phi3:mini` or `llama3.2:1b`
- **Best balance**: `llama3.2` (current choice) or `mistral`
- **Best embeddings**: `nomic-embed-text` (current) or `mxbai-embed-large`